# Merging data and preparing panels

Notebook purpose:
- Load and combine individual tables from the Toetzke et al. (XXXX) linkedin network database
- Load and merge priorly scraped information of IPCEI firms
- Build different data panels for event-study plots and DiD estimation


#### 1. Set-up

In [25]:
import pandas as pd

In [26]:
# paths
from ipcei_spillovers.io import data_path, results_path, load_df, save_df

# data loading pipeline
from ipcei_spillovers.processing_linkedin import run_linkedin_interim_pipeline
from ipcei_spillovers.processing_ipcei import run_ipcei_processing

# logging
from ipcei_spillovers.config_logging import setup_logging
log = setup_logging(filename="analysis.log")

#### 2. Process and load data

a) Run pipelines for linkedin and ipcei data once

In [3]:
# Load linkedin processing pipeline once
tables = run_linkedin_interim_pipeline(version="jan26")

# Convenient handles
firm_master = tables["firm_master"]
partnership_master = tables["partnership_master"]

# Check
print("firm_master:", firm_master.shape)
print("partnership_master:", partnership_master.shape)
firm_master.head()

2026-01-28 16:25:18 | INFO | ipcei | Loading LinkedIn tables for version=jan26
2026-01-28 16:25:31 | INFO | ipcei | Loaded partnerships    shape=(3943633, 6)
2026-01-28 16:25:31 | INFO | ipcei | Loaded firms           shape=(255784, 24)
2026-01-28 16:25:31 | INFO | ipcei | Loaded locations       shape=(437172, 14)
2026-01-28 16:25:31 | INFO | ipcei | Loaded collaborations  shape=(769167, 57)
2026-01-28 16:25:31 | INFO | ipcei | Loaded meta            shape=(828233, 7)
2026-01-28 16:25:36 | INFO | ipcei | Slimmed firms=(255784, 10) meta=(828033, 2) locations=(437172, 11)
2026-01-28 16:25:36 | WARNING | ipcei | Firms has 16002 duplicate Linkedin_name rows -> deduplicating
2026-01-28 16:25:37 | WARNING | ipcei | Meta has 648929 duplicate Linkedin_name rows -> deduplicating
2026-01-28 16:25:37 | INFO | ipcei | Built firm_master shape=(239782, 11)
2026-01-28 16:29:06 | INFO | ipcei | Built partnership_master shape=(3943633, 57) indicators_used=51
2026-01-28 16:29:07 | INFO | ipcei | Saving 

,Linkedin_name,foundingYear,website,name,linkedinUrl,NEW_Linkedin_name,firm_type,sector,industry,meta_update,...,has_headquarter,n_locations_closed,any_location_closed,latest_closingDate,primary_country,primary_city,primary_latitude,primary_longitude,hq_country,hq_city
0,#startupindia,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,...,<NA>,<NA>,<NA>,NaT,NaN,NaN,NaN,NaN,<NA>,<NA>
1,%27albeton%27-algemene-beton-maatschappij-b-v-,1968.0,http://www.albeton.nl,'Albeton' Algemene Beton Maatschappij B.V.,https://www.linkedin.com/company/%27albeton%27...,<NA>,Manufacturer,Built Environment,"Glass, Ceramics and Concrete Manufacturing",2025-12-17 08:40:36,...,1,0,0,NaT,NL,Amsterdam,4.83448,52.4153,NL,Amsterdam
2,%C2%B0celsios,2024.0,<NA>,%C2%B0celsios,https://www.linkedin.com/company/%C2%B0celsios,<NA>,Service Provider,Built Environment,Real Estate,2026-01-16 14:46:28,...,<NA>,<NA>,<NA>,NaT,NaN,NaN,NaN,NaN,<NA>,<NA>
3,%C2%B5avionix-corporation,2015.0,http://www.uavionix.com,uAvionix Corporation,https://www.linkedin.com/company/%C2%B5avionix...,<NA>,Manufacturer,Aviation,Aviation and Aerospace Component Manufacturing,2026-01-14 22:24:19,...,<NA>,<NA>,<NA>,NaT,NaN,NaN,NaN,NaN,<NA>,<NA>
4,%C3%A1balta-special-school,NaN,http://abaltaschool.ie,Ábalta Special School,https://www.linkedin.com/company/%C3%A1balta-s...,<NA>,<NA>,Other Sectors,Primary and Secondary Education,2025-12-17 08:58:52,...,<NA>,<NA>,<NA>,NaT,NaN,NaN,NaN,NaN,<NA>,<NA>


In [27]:
# Run IPCEI processing pipeline once
ipcei_tables = run_ipcei_processing(version="jan26")

# Convenient handles
ipcei_events = ipcei_tables["ipcei_events"]
ipcei_firm_summary = ipcei_tables["ipcei_firm_summary"]

print("ipcei_events:", ipcei_events.shape)
print("ipcei_firm_summary:", ipcei_firm_summary.shape)

ipcei_events.head()

2026-02-13 17:06:54 | INFO | ipcei | Loading raw IPCEI + interim LinkedIn masters (version=jan26)
2026-02-13 17:07:19 | INFO | ipcei | Saved IPCEI interim outputs for version=jan26
ipcei_events: (173, 13)
ipcei_firm_summary: (157, 8)
ipcei_firm_event_panel: (173, 11)


,Linkedin_name,name,country,ipcei_type,ipcei_subtype,ipcei_post_date,ipcei_round,ipcei_round_name,event_id,event_month,in_firm_master,in_partnerships,in_either_master
0,1s1-energy,1s1 Energy,PT,hydrogen,"Production, Utillization",2022-07-15,1,Hy2Tech,1s1-energy|hydrogen|r1,2022-07-01,1,1,1
1,acc-automotive-cells-company,ACC,"DE, FR",battery,Battery cells,2019-12-09,1,Battery value chain 1,acc-automotive-cells-company|battery|r1,2019-12-01,1,1,1
2,acis-complex-kft,ACIS,DE,battery,"Raw and advanced materials, Battery systems",2021-01-26,2,Battery value chain 2,acis-complex-kft|battery|r2,2021-01-01,0,0,0
3,advent-technologies-inc,Advent Technologies,GR,hydrogen,"Production, Utillization",2022-07-15,1,Hy2Tech,advent-technologies-inc|hydrogen|r1,2022-07-01,1,1,1
4,ae-elemental,Elemental,"PL, LU",battery,Recycling and sustainability,2019-12-09,1,Battery value chain 1,ae-elemental|battery|r1,2019-12-01,1,1,1


b) Load data if processed before

In [33]:
# Load cleaned data if pipeline ran once

ipcei_events = load_df(
    "interim", "ipcei", "ipcei_events_jan26.csv", low_memory=False
)
ipcei_firm_summary = load_df(
    "interim", "ipcei", "ipcei_firm_summary_jan26.csv", low_memory=False
)
firm_master = load_df(
    "interim", "linkedin", "firm_master_jan26.csv", low_memory=False
)
partnership_master = load_df(
    "interim", "linkedin", "partnership_master_jan26.csv", low_memory=False
)

print("ipcei_events:", ipcei_events.shape)
print("ipcei_firm_summary:", ipcei_firm_summary.shape)

ipcei_events: (173, 13)
ipcei_firm_summary: (157, 8)


#### 3. Merge and prepare different panels

In [35]:
VERSION = "jan26"

# ============================================================
# 1) Helpers
# ============================================================
def _to_dt(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")

def _to_month(s: pd.Series) -> pd.Series:
    # returns month timestamp (YYYY-MM-01)
    return _to_dt(s).dt.to_period("M").dt.to_timestamp()

def _req(df: pd.DataFrame, cols: list[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")

def _event_grid(row):
    g = row["g"]
    months = pd.date_range(g - pd.DateOffset(months=WINDOW), g + pd.DateOffset(months=WINDOW), freq="MS")
    out = pd.DataFrame({"event_id": row["event_id"], "t": months})
    out["Linkedin_name"] = row["Linkedin_name"]
    out["g"] = row["g"]
    out["ipcei_type"] = row["ipcei_type"]
    out["ipcei_round"] = row["ipcei_round"]
    out["ipcei_round_name"] = row["ipcei_round_name"]
    out["country"] = row["country"]
    return out

a) Plain merges

In [30]:
# ============================================================
# 2) Firm-level merge (1 row per firm)
# ============================================================
firm_master_ipcei = firm_master.merge(
    ipcei_firm_summary,
    on="Linkedin_name",
    how="left",
    validate="1:1",
)
firm_master_ipcei["treated_any"] = firm_master_ipcei["treated_any"].fillna(0).astype(int)

# ============================================================
# 3) Firm × event table (1 row per IPCEI event)
# ============================================================
ipcei_events2 = ipcei_events.copy()

_req(
    ipcei_events2,
    ["Linkedin_name", "event_id", "ipcei_post_date", "event_month", "ipcei_type", "ipcei_round"],
    "ipcei_events",
)

# ensure types
ipcei_events2["ipcei_post_date"] = _to_dt(ipcei_events2["ipcei_post_date"])
ipcei_events2["event_month"] = _to_month(ipcei_events2["event_month"])  # safe even if already month-ts

firm_master_ipcei_events = firm_master.merge(
    ipcei_events2,
    on="Linkedin_name",
    how="right",     # keeps only treated/event firms
    validate="1:m",
)

b) Edge-level stack for IPCEI plots

In [31]:
# ============================================================
# 4) Event-specific edge stack
#    One row per (edge-row × event) where the event_firm is source OR target
# ============================================================
p = partnership_master.copy()
_req(p, ["post_date", "source", "target", "collaboration_id"], "partnership_master")

p["post_date"] = _to_dt(p["post_date"])
p = p.dropna(subset=["post_date", "source", "target", "collaboration_id"]).copy()
p["post_month"] = _to_month(p["post_date"])

# keep only event columns we need (avoid collisions)
ev = ipcei_events2[[
    "Linkedin_name",
    "event_id",
    "ipcei_type",
    "ipcei_round",
    "ipcei_round_name",
    "country",
    "ipcei_post_date",
    "event_month",
]].copy()

# create cohort month "g" (canonical staggered DiD notation)
ev["g"] = ev["event_month"]

# ---- source-event stack ----
src_stack = (
    p.merge(ev.rename(columns={"Linkedin_name": "source"}), on="source", how="inner")
     .assign(
         event_endpoint="source",
         event_firm=lambda d: d["source"],
         other_firm=lambda d: d["target"],
     )
)

# ---- target-event stack ----
tgt_stack = (
    p.merge(ev.rename(columns={"Linkedin_name": "target"}), on="target", how="inner")
     .assign(
         event_endpoint="target",
         event_firm=lambda d: d["target"],
         other_firm=lambda d: d["source"],
     )
)

edge_event_stack = pd.concat([src_stack, tgt_stack], ignore_index=True)

# canonical time variable "t"
edge_event_stack["t"] = edge_event_stack["post_month"]

# event_time in months (t - g)
edge_event_stack["event_time"] = (
    edge_event_stack["t"].dt.to_period("M") - edge_event_stack["g"].dt.to_period("M")
).apply(lambda x: x.n)

edge_event_stack["post_event"] = (edge_event_stack["event_time"] >= 0).astype(int)

# Drop duplicates if the same (event_firm,event_id) appears twice on same collab-month
edge_event_stack = edge_event_stack.drop_duplicates(
    subset=["collaboration_id", "post_month", "event_id", "event_firm"],
    keep="first",
)

# ============================================================
# 5) Endpoint-summary edge merge for descriptives
# ============================================================
endpoint_lookup = ipcei_firm_summary.copy()

src = endpoint_lookup.add_prefix("source_").rename(columns={"source_Linkedin_name": "source"})
tgt = endpoint_lookup.add_prefix("target_").rename(columns={"target_Linkedin_name": "target"})

partnership_master_ipcei = (
    p.drop(columns=["post_month"], errors="ignore")
     .merge(src, on="source", how="left", validate="m:1")
     .merge(tgt, on="target", how="left", validate="m:1")
)

for c in ["source_treated_any", "target_treated_any"]:
    partnership_master_ipcei[c] = partnership_master_ipcei[c].fillna(0).astype(int)

partnership_master_ipcei["any_treated"] = (
    partnership_master_ipcei["source_treated_any"] | partnership_master_ipcei["target_treated_any"]
).astype(int)

partnership_master_ipcei["both_treated"] = (
    partnership_master_ipcei["source_treated_any"] & partnership_master_ipcei["target_treated_any"]
).astype(int)

# reference only
for c in ["source_first_ipcei_date", "target_first_ipcei_date"]:
    if c in partnership_master_ipcei.columns:
        partnership_master_ipcei[c] = pd.to_datetime(partnership_master_ipcei[c], errors="coerce")

partnership_master_ipcei["edge_treat_date_first"] = partnership_master_ipcei[
    ["source_first_ipcei_date", "target_first_ipcei_date"]
].min(axis=1, skipna=True)

# ============================================================
# 6) Save
# ============================================================
save_df(firm_master_ipcei,        "interim", "combined", f"firm_master_ipcei_{VERSION}.csv")
save_df(firm_master_ipcei_events, "interim", "combined", f"firm_master_ipcei_events_{VERSION}.csv")
save_df(partnership_master_ipcei, "interim", "combined", f"partnership_master_ipcei_{VERSION}.csv")
save_df(edge_event_stack,         "interim", "combined", f"edge_event_stack_ipcei_{VERSION}.csv")

print("Saved:")
print("  firm_master_ipcei:", firm_master_ipcei.shape)
print("  firm_master_ipcei_events:", firm_master_ipcei_events.shape)
print("  partnership_master_ipcei:", partnership_master_ipcei.shape)
print("  edge_event_stack:", edge_event_stack.shape)

print("\nEdge-level panel unique events:", edge_event_stack["event_id"].nunique())
print("Edge-level panel unique event firms:", edge_event_stack["event_firm"].nunique())

# CHECK: does each event_id have exactly one g
tmp = edge_event_stack.groupby("event_id")["g"].nunique()
print("Event_ids with >1 cohort month g:", int((tmp > 1).sum()))

Saved:
  firm_master_ipcei: (239782, 31)
  firm_master_ipcei_events: (173, 36)
  partnership_master_ipcei: (3943633, 74)
  edge_level_panel: (86597, 72)

Edge-level panel unique events: 160
Edge-level panel unique event firms: 145
Event_ids with >1 cohort month g: 0


c) Firm-months outcomes for later panels

In [36]:
# =========================
# Inputs expected in memory
# =========================

_req(partnership_master, ["post_date", "source", "target", "collaboration_id"], "partnership_master")
_req(ipcei_events, ["Linkedin_name", "event_id", "event_month", "ipcei_post_date", "ipcei_type", "ipcei_round"], "ipcei_events")

# =========================
# Firm-month outcomes (unique collaborations per month)
# =========================
p = partnership_master.copy()
p["post_date"] = _to_dt(p["post_date"])
p = p.dropna(subset=["post_date", "source", "target", "collaboration_id"]).copy()
p["month"] = _to_month(p["post_date"])

# Stack endpoints so "firm" = source OR target
src = p[["source", "month", "collaboration_id"]].rename(columns={"source": "Linkedin_name"})
tgt = p[["target", "month", "collaboration_id"]].rename(columns={"target": "Linkedin_name"})

ft = pd.concat([src, tgt], ignore_index=True)
ft = ft.dropna(subset=["Linkedin_name", "month", "collaboration_id"])

firm_month_outcomes = (
    ft.groupby(["Linkedin_name", "month"])["collaboration_id"]
      .nunique()
      .reset_index(name="n_collabs")
)

print("firm_month_outcomes:", firm_month_outcomes.shape)
firm_month_outcomes.head()


# =========================
# Save
# =========================
save_df(firm_month_outcomes, "interim", "combined", f"firm_month_outcomes_{VERSION}.csv")

firm_month_outcomes: (986263, 3)


PosixPath('/Users/go82gax/Documents/Projekte/IPCEI/Analyse/PythonProject/ipcei-spillovers/src/data/interim/combined/firm_month_outcomes_jan26.csv')

#### 4. Inspect merge and data quality

In [14]:
# ----------------------------
# Load data
# ----------------------------

event_panel = load_df(
    "interim", "combined", "edge_event_stack_ipcei_jan26.csv", low_memory=False
)
firm_master_ipcei_events = load_df(
    "interim", "combined", "firm_master_ipcei_events_jan26.csv", low_memory=False
)
firm_master_ipcei = load_df(
    "interim", "combined", "firm_master_ipcei_jan26.csv", low_memory=False
)
partnership_master_ipcei = load_df(
    "interim", "combined", "partnership_master_ipcei_jan26.csv", low_memory=False
)

print(f"event_panel: {event_panel.shape}")
print(f"firm_master_ipcei: {firm_master_ipcei.shape}")
print(f"partnership_master_ipcei: {partnership_master_ipcei.shape}")
print(f"firm_master_ipcei_events: {firm_master_ipcei_events.shape}")

event_panel: (86732, 72)
firm_panel: (8477, 11)
firm_master_ipcei: (239782, 31)
partnership_master_ipcei: (3943633, 74)
firm_master_ipcei_events: (173, 36)


In [16]:
# ----------------------------
# Configuration
# ----------------------------
# Create audit subdirectory in results/tables
AUDIT_DIR = results_path("tables", "audit")
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Helpers
# ----------------------------
def _norm(s: pd.Series) -> pd.Series:
    """Normalize string series for comparison"""
    return s.astype("string").str.strip()

def _split_countries(s: pd.Series) -> pd.Series:
    """Split comma-separated country strings into lists"""
    return (
        s.dropna()
         .astype("string")
         .str.upper()
         .str.split(",")
         .apply(lambda lst: [x.strip() for x in lst])
    )

def save_audit_list(df: pd.DataFrame, filename: str, description: str):
    """Save audit list to CSV with metadata"""
    if len(df) > 0:
        output_path = AUDIT_DIR / filename
        df.to_csv(output_path, index=False)
        print(f"  ✓ Saved {len(df)} firms to {filename}")
    else:
        print(f"  ✓ No issues found for {description}")

In [24]:
# ----------------------------
# AUDIT CHECKS
# ----------------------------

print("\n" + "=" * 80)
print("IPCEI DATA QUALITY AUDIT")
print("=" * 80)

# Initialize results dictionary for summary table
audit_summary = {}

# Get base counts
ipcei_firms_all = firm_master_ipcei_events["Linkedin_name"].unique()
n_total_ipcei_firms = len(ipcei_firms_all)

firm_master_names = set(_norm(firm_master_ipcei["Linkedin_name"]).dropna())
partnership_names = set(
    pd.concat([
        partnership_master_ipcei["source"],
        partnership_master_ipcei["target"],
    ])
    .dropna()
    .pipe(_norm)
)

print(f"\nTotal unique IPCEI firms in events: {n_total_ipcei_firms}")

# ----------------------------
# 1. Basic IPCEI Statistics
# ----------------------------
print("\n" + "─" * 80)
print("1. BASIC IPCEI STATISTICS")
print("─" * 80)

# Count by technology type
battery_firms = firm_master_ipcei_events[
    firm_master_ipcei_events["ipcei_type"].str.lower() == "battery"
]["Linkedin_name"].unique()

hydrogen_firms = firm_master_ipcei_events[
    firm_master_ipcei_events["ipcei_type"].str.lower() == "hydrogen"
]["Linkedin_name"].unique()

both_tech_firms = set(battery_firms) & set(hydrogen_firms)

audit_summary["total_ipcei_firms"] = {
    "count": n_total_ipcei_firms,
    "pct": 100.0
}

audit_summary["battery_only"] = {
    "count": len(set(battery_firms) - both_tech_firms),
    "pct": len(set(battery_firms) - both_tech_firms) / n_total_ipcei_firms * 100
}

audit_summary["hydrogen_only"] = {
    "count": len(set(hydrogen_firms) - both_tech_firms),
    "pct": len(set(hydrogen_firms) - both_tech_firms) / n_total_ipcei_firms * 100
}

audit_summary["both_technologies"] = {
    "count": len(both_tech_firms),
    "pct": len(both_tech_firms) / n_total_ipcei_firms * 100
}

print(f"  Battery only: {audit_summary['battery_only']['count']} ({audit_summary['battery_only']['pct']:.1f}%)")
print(f"  Hydrogen only: {audit_summary['hydrogen_only']['count']} ({audit_summary['hydrogen_only']['pct']:.1f}%)")
print(f"  Both technologies: {audit_summary['both_technologies']['count']} ({audit_summary['both_technologies']['pct']:.1f}%)")

# Count firms with multiple rounds
multi_round_firms = (
    firm_master_ipcei_events
    .groupby("Linkedin_name")["event_id"]
    .nunique()
    .reset_index(name="n_rounds")
)

n_multi_round = (multi_round_firms["n_rounds"] > 1).sum()

audit_summary["multiple_rounds"] = {
    "count": n_multi_round,
    "pct": n_multi_round / n_total_ipcei_firms * 100
}

print(f"  Multiple rounds: {n_multi_round} ({audit_summary['multiple_rounds']['pct']:.1f}%)")

# Save list of multi-round firms
multi_round_list = firm_master_ipcei_events[
    firm_master_ipcei_events["Linkedin_name"].isin(
        multi_round_firms[multi_round_firms["n_rounds"] > 1]["Linkedin_name"]
    )
].sort_values(["Linkedin_name", "ipcei_post_date"])[
    ["Linkedin_name", "event_id", "ipcei_type", "ipcei_round", "ipcei_post_date", "country"]
]

save_audit_list(
    multi_round_list,
    "01_firms_multiple_rounds.csv",
    "firms with multiple rounds"
)

# ----------------------------
# 2. Unmatched to firm_master
# ----------------------------
print("\n" + "─" * 80)
print("2. IPCEI FIRMS NOT IN LINKEDIN DATA")  # CHANGED
print("─" * 80)

unmatched_firm_master = firm_master_ipcei_events[
    ~firm_master_ipcei_events["Linkedin_name"].isin(firm_master_names)
].copy()

n_unmatched_firm_master = unmatched_firm_master["Linkedin_name"].nunique()

audit_summary["unmatched_firm_master"] = {
    "count": n_unmatched_firm_master,
    "pct": n_unmatched_firm_master / n_total_ipcei_firms * 100
}

print(f"  Unmatched: {n_unmatched_firm_master} ({audit_summary['unmatched_firm_master']['pct']:.1f}%)")

unmatched_firm_master_list = unmatched_firm_master.sort_values("Linkedin_name")[
    ["Linkedin_name", "event_id", "ipcei_type", "ipcei_round", "country"]
].drop_duplicates()

save_audit_list(
    unmatched_firm_master_list,
    "02_unmatched_linkedin_data.csv",  # CHANGED filename
    "firms not in LinkedIn data"  # CHANGED description
)

# removes unmatched firms from dataframe for further audits
firm_master_ipcei_events = firm_master_ipcei_events[
    firm_master_ipcei_events["Linkedin_name"].isin(firm_master_names)
].copy()

# ----------------------------
# 3. Unmatched to partnership_master
# ----------------------------
print("\n" + "─" * 80)
print("3. IPCEI FIRMS WITH NO COLLABORATIONS IN LINKEDIN")  # CHANGED
print("─" * 80)

# Only check firms that ARE in firm_master
unmatched_partnerships = firm_master_ipcei_events[
    firm_master_ipcei_events["Linkedin_name"].isin(firm_master_names) &
    ~firm_master_ipcei_events["Linkedin_name"].isin(partnership_names)
].copy()

n_unmatched_partnerships = unmatched_partnerships["Linkedin_name"].nunique()

audit_summary["unmatched_partnerships"] = {
    "count": n_unmatched_partnerships,
    "pct": n_unmatched_partnerships / n_total_ipcei_firms * 100
}

print(f"  In firm_master but no collaborations: {n_unmatched_partnerships} ({audit_summary['unmatched_partnerships']['pct']:.1f}%)")

unmatched_partnerships_list = unmatched_partnerships.sort_values("Linkedin_name")[
    ["Linkedin_name", "event_id", "ipcei_type", "ipcei_round", "country"]
].drop_duplicates()

save_audit_list(
    unmatched_partnerships_list,
    "03_no_collaborations_linkedin.csv",  # CHANGED filename
    "firms with no collaborations in LinkedIn"  # CHANGED description
)

# ----------------------------
# 4. No matching technology collaboration
# ----------------------------
print("\n" + "─" * 80)
print("4. IPCEI FIRMS WITH NO MATCHING TECH COLLABORATION")
print("─" * 80)

# Only check firms that ARE in partnerships
ipcei_in_partnerships = firm_master_ipcei_events[
    firm_master_ipcei_events["Linkedin_name"].isin(partnership_names)
].copy()

# Aggregate collaboration flags by firm (check both source and target)
collab_flags_source = (
    partnership_master_ipcei
    .groupby("source")[["Battery", "Hydrogen"]]
    .max()
    .reset_index()
    .rename(columns={"source": "Linkedin_name"})
)

collab_flags_target = (
    partnership_master_ipcei
    .groupby("target")[["Battery", "Hydrogen"]]
    .max()
    .reset_index()
    .rename(columns={"target": "Linkedin_name"})
)

# Combine source and target flags (firm has collab if it appears in either)
collab_flags = pd.concat([collab_flags_source, collab_flags_target]).groupby("Linkedin_name").max().reset_index()

ipcei_in_partnerships = ipcei_in_partnerships.merge(
    collab_flags, on="Linkedin_name", how="left"
)

# Fill NaN with 0 (no collaboration)
ipcei_in_partnerships["Battery"] = ipcei_in_partnerships["Battery"].fillna(0)
ipcei_in_partnerships["Hydrogen"] = ipcei_in_partnerships["Hydrogen"].fillna(0)

# Check mismatches
no_matching_collab = ipcei_in_partnerships[
    (
        (ipcei_in_partnerships["ipcei_type"].str.lower() == "battery")
        & (ipcei_in_partnerships["Battery"] != 1)
    )
    |
    (
        (ipcei_in_partnerships["ipcei_type"].str.lower() == "hydrogen")
        & (ipcei_in_partnerships["Hydrogen"] != 1)
    )
].copy()

n_no_matching_collab = no_matching_collab["Linkedin_name"].nunique()

audit_summary["no_matching_tech_collab"] = {
    "count": n_no_matching_collab,
    "pct": n_no_matching_collab / n_total_ipcei_firms * 100
}

print(f"  No matching collab: {n_no_matching_collab} ({audit_summary['no_matching_tech_collab']['pct']:.1f}%)")

no_matching_collab_list = no_matching_collab.sort_values("Linkedin_name")[
    ["Linkedin_name", "event_id", "ipcei_type", "ipcei_round", "Battery", "Hydrogen"]
].drop_duplicates()

save_audit_list(
    no_matching_collab_list,
    "04_no_matching_tech_collab.csv",
    "firms with no matching tech collaboration"
)

# ----------------------------
# 5. Missing firm characteristics
# ----------------------------
print("\n" + "─" * 80)
print("5. IPCEI FIRMS WITH MISSING CHARACTERISTICS")
print("─" * 80)

# Only check firms that ARE in firm_master
ipcei_in_firm_master = firm_master_ipcei[
    firm_master_ipcei["treated_any"] == 1
].copy()

cols_check = ["firm_type", "sector", "industry", "staffCount", "foundingYear"]

# CHANGED: Store individual missing stats in audit_summary
missing_breakdown = {}
for col in cols_check:
    if col in ipcei_in_firm_master.columns:
        n_missing = ipcei_in_firm_master[col].isna().sum()
        missing_breakdown[col] = {
            "count": n_missing,
            "pct": n_missing / n_total_ipcei_firms * 100  # Changed denominator to total IPCEI firms
        }

        # ADD EACH TO AUDIT SUMMARY
        audit_summary[f"missing_{col}"] = missing_breakdown[col]

        print(f"  Missing {col}: {n_missing} ({missing_breakdown[col]['pct']:.1f}%)")
    else:
        print(f"  ⚠ Column '{col}' not found in firm_master_ipcei")

# Overall: firms missing ANY characteristic
available_cols = [c for c in cols_check if c in ipcei_in_firm_master.columns]
if available_cols:
    missing_any = ipcei_in_firm_master[
        ipcei_in_firm_master[available_cols].isna().any(axis=1)
    ].copy()

    n_missing_any = len(missing_any)

    audit_summary["missing_any_characteristic"] = {
        "count": n_missing_any,
        "pct": n_missing_any / n_total_ipcei_firms * 100
    }

    print(f"  Missing ANY characteristic: {n_missing_any} ({audit_summary['missing_any_characteristic']['pct']:.1f}%)")

    # Create detailed list showing which fields are missing
    missing_any_list = missing_any.copy()
    missing_any_list["missing_fields"] = missing_any_list[available_cols].isna().apply(
        lambda row: ", ".join([col for col, is_missing in row.items() if is_missing]),
        axis=1
    )

    cols_to_export = ["Linkedin_name"] + available_cols + ["missing_fields"]
    missing_any_list = missing_any_list[cols_to_export].sort_values("Linkedin_name")

    save_audit_list(
        missing_any_list,
        "05_missing_firm_characteristics.csv",
        "firms with missing characteristics"
    )
else:
    print("  ⚠ No characteristic columns found to check")
    audit_summary["missing_any_characteristic"] = {"count": 0, "pct": 0.0}

# ----------------------------
# 6. Country mismatch
# ----------------------------
print("\n" + "─" * 80)
print("6. IPCEI COUNTRY VS FIRM_MASTER COUNTRY MISMATCH")
print("─" * 80)

# IPCEI side: country already clean, split into list
ipcei_countries = (
    firm_master_ipcei_events[["Linkedin_name", "country"]]
    .dropna()
    .assign(
        ipcei_country_list=lambda d: d["country"].apply(
            lambda x: [c.strip().upper() for c in x.split(",")]
        )
    )
)

# Check which country columns exist in firm master
country_cols = []
for col in ["primary_country", "countries"]:
    if col in firm_master_ipcei.columns:
        country_cols.append(col)

if country_cols:
    # Firm side: SPLIT the countries string into tokens
    firm_countries = (
        firm_master_ipcei
        .loc[
            firm_master_ipcei["treated_any"] == 1,
            ["Linkedin_name"] + country_cols
        ]
        .assign(
            firm_country_list=lambda d: d[country_cols].apply(
                lambda r: [
                    c.strip().upper()
                    for val in r
                    if pd.notna(val)
                    for c in val.split(",")
                ],
                axis=1
            )
        )
    )

    country_check = ipcei_countries.merge(
        firm_countries, on="Linkedin_name", how="left"
    )

    # Check if ANY IPCEI country is contained in firm's countries
    def check_country_overlap(row):
        if not isinstance(row["firm_country_list"], list):
            return False
        if not isinstance(row["ipcei_country_list"], list):
            return True

        return any(c in row["firm_country_list"] for c in row["ipcei_country_list"])

    country_check["has_overlap"] = country_check.apply(check_country_overlap, axis=1)

    country_mismatch = country_check[~country_check["has_overlap"]].copy()

    n_country_mismatch = country_mismatch["Linkedin_name"].nunique()

    audit_summary["country_mismatch"] = {
        "count": n_country_mismatch,
        "pct": n_country_mismatch / n_total_ipcei_firms * 100
    }

    print(f"  Country mismatch: {n_country_mismatch} ({audit_summary['country_mismatch']['pct']:.1f}%)")

    # Export list
    export_cols = ["Linkedin_name", "country"] + country_cols
    country_mismatch_list = (
        country_mismatch[export_cols]
        .drop_duplicates()
        .sort_values("Linkedin_name")
    )

    save_audit_list(
        country_mismatch_list,
        "06_country_mismatch.csv",
        "firms with country mismatch"
    )
else:
    print("  ⚠ No country columns found in firm_master_ipcei")
    audit_summary["country_mismatch"] = {"count": 0, "pct": 0.0}

# ----------------------------
# 7. Create Summary Table for LaTeX
# ----------------------------
print("\n" + "─" * 80)
print("7. GENERATING LATEX SUMMARY TABLE")
print("─" * 80)

# Build summary dataframe
summary_rows = [
    ("Total IPCEI Firms", audit_summary["total_ipcei_firms"]["count"], audit_summary["total_ipcei_firms"]["pct"]),
    ("\\quad Battery only", audit_summary["battery_only"]["count"], audit_summary["battery_only"]["pct"]),
    ("\\quad Hydrogen only", audit_summary["hydrogen_only"]["count"], audit_summary["hydrogen_only"]["pct"]),
    ("\\quad Both technologies", audit_summary["both_technologies"]["count"], audit_summary["both_technologies"]["pct"]),
    ("\\quad Multiple funding rounds", audit_summary["multiple_rounds"]["count"], audit_summary["multiple_rounds"]["pct"]),
    ("\\midrule", "", ""),
    ("\\textbf{Data Quality Issues}", "", ""),
    ("Not in LinkedIn data", audit_summary["unmatched_firm_master"]["count"], audit_summary["unmatched_firm_master"]["pct"]),  # CHANGED
    ("No collaborations in LinkedIn", audit_summary["unmatched_partnerships"]["count"], audit_summary["unmatched_partnerships"]["pct"]),  # CHANGED
    ("No matching tech collaboration", audit_summary["no_matching_tech_collab"]["count"], audit_summary["no_matching_tech_collab"]["pct"]),
    ("Country mismatch", audit_summary["country_mismatch"]["count"], audit_summary["country_mismatch"]["pct"]),
    ("Missing ANY characteristic", audit_summary["missing_any_characteristic"]["count"], audit_summary["missing_any_characteristic"]["pct"]),  # CHANGED label
]

# Add individual missing rows
if "missing_firm_type" in audit_summary:
    summary_rows.append(("\\quad Missing firm type", audit_summary["missing_firm_type"]["count"], audit_summary["missing_firm_type"]["pct"]))
if "missing_sector" in audit_summary:
    summary_rows.append(("\\quad Missing sector", audit_summary["missing_sector"]["count"], audit_summary["missing_sector"]["pct"]))
if "missing_industry" in audit_summary:
    summary_rows.append(("\\quad Missing industry", audit_summary["missing_industry"]["count"], audit_summary["missing_industry"]["pct"]))
if "missing_staffCount" in audit_summary:
    summary_rows.append(("\\quad Missing staff count", audit_summary["missing_staffCount"]["count"], audit_summary["missing_staffCount"]["pct"]))
if "missing_foundingYear" in audit_summary:
    summary_rows.append(("\\quad Missing founding year", audit_summary["missing_foundingYear"]["count"], audit_summary["missing_foundingYear"]["pct"]))

summary_df = pd.DataFrame(summary_rows, columns=["Category", "Count", "Percentage"])

# Create LaTeX table
latex_output = []
latex_output.append("\\begin{table}[htbp]")
latex_output.append("\\centering")
latex_output.append("\\caption{IPCEI Data Quality Audit}")
latex_output.append("\\label{tab:ipcei_audit}")
latex_output.append("\\begin{tabular}{lrr}")
latex_output.append("\\toprule")
latex_output.append("Category & Count & \\% \\\\")
latex_output.append("\\midrule")

for _, row in summary_df.iterrows():
    if row["Category"] == "\\midrule":
        latex_output.append("\\midrule")
    elif row["Count"] == "":
        latex_output.append(f"{row['Category']} & & \\\\")
    else:
        latex_output.append(f"{row['Category']} & {row['Count']:,} & {row['Percentage']:.1f}\\% \\\\")

latex_output.append("\\bottomrule")
latex_output.append("\\end{tabular}")
latex_output.append("\\begin{tablenotes}")
latex_output.append("\\small")
latex_output.append("\\item Note: Percentages calculated relative to total IPCEI firms (N={:,}).".format(n_total_ipcei_firms))
latex_output.append("\\item Detailed firm lists for each category available in supplementary files.")
latex_output.append("\\end{tablenotes}")
latex_output.append("\\end{table}")

latex_table = "\n".join(latex_output)

# Save LaTeX table
latex_path = AUDIT_DIR / "audit_summary_table.tex"
with open(latex_path, "w") as f:
    f.write(latex_table)

print(f"  ✓ Saved LaTeX table to {latex_path}")

# Also save as CSV for reference
summary_csv_path = AUDIT_DIR / "audit_summary_table.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"  ✓ Saved summary CSV to {summary_csv_path}")

# ----------------------------
# 8. Print LaTeX Table
# ----------------------------
print("\n" + "=" * 80)
print("LATEX TABLE OUTPUT")
print("=" * 80)
print("\n" + latex_table)

print("\n" + "=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)
print(f"\nAll outputs saved to: {AUDIT_DIR.absolute()}")
print("\nGenerated files:")
print("  - audit_summary_table.tex (for LaTeX)")
print("  - audit_summary_table.csv (for reference)")
print("  - 01_firms_multiple_rounds.csv")
print("  - 02_unmatched_firm_master.csv")
print("  - 03_unmatched_partnerships.csv")
print("  - 04_no_matching_tech_collab.csv")
print("  - 05_missing_firm_characteristics.csv")
print("  - 06_country_mismatch.csv")


IPCEI DATA QUALITY AUDIT

Total unique IPCEI firms in events: 150

────────────────────────────────────────────────────────────────────────────────
1. BASIC IPCEI STATISTICS
────────────────────────────────────────────────────────────────────────────────
  Battery only: 42 (28.0%)
  Hydrogen only: 103 (68.7%)
  Both technologies: 5 (3.3%)
  Multiple rounds: 14 (9.3%)
  ✓ Saved 30 firms to 01_firms_multiple_rounds.csv

────────────────────────────────────────────────────────────────────────────────
2. IPCEI FIRMS NOT IN LINKEDIN DATA
────────────────────────────────────────────────────────────────────────────────
  Unmatched: 0 (0.0%)
  ✓ No issues found for firms not in LinkedIn data

────────────────────────────────────────────────────────────────────────────────
3. IPCEI FIRMS WITH NO COLLABORATIONS IN LINKEDIN
────────────────────────────────────────────────────────────────────────────────
  In firm_master but no collaborations: 5 (3.3%)
  ✓ Saved 6 firms to 03_no_collaborations_li